# SPair-71k: verify and compare results

This notebook helps you **inspect the dataset**, **run PCK evaluations** with the same logic as `scripts/eval_baseline.py`, and **compare** multiple backbones or settings (e.g. baseline vs fine-tuned vs window soft-argmax).

**Project rules (see `docs/info.md`):**
- Use split **`test`** for **reported** final numbers; **`val`** for development and model selection; **`train`** only for training scripts.
- Window Soft-Argmax is **inference-only** (no retraining).

**Dataset:** place SPair-71k under `data/SPair-71k/` (or set `SPAIR_ROOT`), then run `python scripts/verify_dataset.py` once to confirm layout and loading.

**How to use:** edit the **configuration** cell below (paths, splits, experiment list), then run cells top-to-bottom. Heavy cells are marked; use `LIMIT_PAIRS > 0` for quick smoke tests.

In [ ]:
# =============================================================================
# CONFIGURATION — edit this cell to match your machine and experiments
# =============================================================================

from __future__ import annotations

import os
import sys
from pathlib import Path


def _find_repo_root() -> Path:
    """Locate the project root (folder containing `pyproject.toml`) from `getcwd()`."""
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "pyproject.toml").is_file():
            return p
    return cwd


REPO_ROOT = _find_repo_root()
# If imports fail later, you can prepend the repo to `sys.path` manually:
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# --- Dataset -----------------------------------------------------------------
# Leave None to use SPAIR_ROOT / DATASET_ROOT / default `<repo>/data/SPair-71k`.
SPAIR_ROOT: str | None = None

# --- Device ------------------------------------------------------------------
# "cuda", "cpu", or None for auto
DEVICE: str | None = None

# --- Evaluation --------------------------------------------------------------
# PCK thresholds (relative to bbox size, same as CLI)
ALPHAS: tuple[float, ...] = (0.05, 0.1, 0.15)

# If > 0, only the first N pairs are evaluated per run (fast debugging).
LIMIT_PAIRS: int = 0

# Default split for the comparison loop below (override per experiment in EXPERIMENTS).
DEFAULT_SPLIT: str = "val"

# Output directory for JSON/CSV (under repo; `runs/` is gitignored)
OUTPUT_DIR = REPO_ROOT / "runs" / "notebook_exports"

# --- Official backbone weights (set paths you have locally) ------------------
# DINOv2/DINOv3: optional; if None, DINOv2 hub may download when possible.
DINOV2_WEIGHTS: str | None = None
DINOV3_WEIGHTS: str | None = None
SAM_CHECKPOINT: str | None = None

# --- Experiment list (edit freely) ------------------------------------------
# Each entry is a dict passed to `EvalRunSpec(**entry, ...)`.
# Keys must match `evaluation.experiment_runner.EvalRunSpec` fields.
# Add `checkpoint` paths to compare fine-tuned / LoRA models.

EXPERIMENTS: list[dict] = [
    {
        "name": "dinov2_baseline",
        "backbone": "dinov2_vitb14",
        "split": DEFAULT_SPLIT,
        "dinov2_weights": DINOV2_WEIGHTS,
        "use_window_soft_argmax": False,
        "limit": LIMIT_PAIRS,
    },
    {
        "name": "dinov2_wsa",
        "backbone": "dinov2_vitb14",
        "split": DEFAULT_SPLIT,
        "dinov2_weights": DINOV2_WEIGHTS,
        "use_window_soft_argmax": True,
        "wsa_window": 5,
        "wsa_temperature": 1.0,
        "limit": LIMIT_PAIRS,
    },
    # --- More templates (uncomment and set paths) ---
    # {
    #     "name": "dinov2_finetuned",
    #     "backbone": "dinov2_vitb14",
    #     "split": DEFAULT_SPLIT,
    #     "dinov2_weights": DINOV2_WEIGHTS,
    #     "checkpoint": str(REPO_ROOT / "checkpoints" / "dinov2_vitb14_lastblocks2_best.pt"),
    #     "limit": LIMIT_PAIRS,
    # },
    # {
    #     "name": "sam_baseline",
    #     "backbone": "sam_vit_b",
    #     "split": DEFAULT_SPLIT,
    #     "sam_checkpoint": SAM_CHECKPOINT,
    #     "limit": LIMIT_PAIRS,
    # },
]

## Imports and optional dependencies

The project package must be importable (recommended: `pip install -e .` from the repo root).

**Backbones:** DINOv2, DINOv3, and SAM code live under `models/dinov2/`, `models/dinov3/`, `models/sam/` (no Hugging Face, see `docs/info.md`). Download pretrained weights separately.

Optional: `pip install -e ".[notebook]"` for Jupyter + pandas (tables and bar charts).

In [ ]:
import json
from typing import Any, Optional

import matplotlib.pyplot as plt
import numpy as np
import torch

# Optional: pandas for tidy tables (install: pip install -e ".[notebook]")
try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from IPython.display import display
except ImportError:
    def display(obj: Any) -> None:
        print(obj)

from data.paths import resolve_spair_root
from data.dataset import PreprocessMode, SPair71kPairDataset, visualize_pair
from evaluation.experiment_runner import EvalRunSpec, metrics_rows_for_table, run_comparison_batch

print("Repo root:", REPO_ROOT)
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## Environment and dataset checks

Verifies that SPair-71k resolves and reports pair counts for each split.

In [ ]:
spair_path = resolve_spair_root(SPAIR_ROOT)
assert os.path.isdir(spair_path), f"SPair-71k not found at: {spair_path}"

for split in ("train", "val", "test"):
    ds = SPair71kPairDataset(
        spair_root=spair_path,
        split=split,
        preprocess=PreprocessMode.FIXED_RESIZE,
        output_size_hw=(784, 784),
        normalize=True,
        photometric_augment=None,
    )
    print(f"split={split:<5}  pairs={len(ds)}")

print("SPAIR_ROOT resolved to:", spair_path)

## Visual sanity check (one pair)

Displays a resized source/target pair from the dataloader (no forward pass).

In [ ]:
_preview_ds = SPair71kPairDataset(
    spair_root=spair_path,
    split="val",
    preprocess=PreprocessMode.FIXED_RESIZE,
    output_size_hw=(784, 784),
    normalize=True,
    photometric_augment=None,
)
_sample = _preview_ds[0]
visualize_pair(_sample["src_img"], _sample["tgt_img"], title="val[0] (FIXED_RESIZE 784)")

## Run PCK evaluation (may be slow / GPU-heavy)

Builds `EvalRunSpec` objects from `EXPERIMENTS` and runs them **sequentially** (same code path as `scripts/eval_baseline.py` via `evaluation.experiment_runner`).

**Tip:** set `LIMIT_PAIRS` (configuration cell) to a small number first.

In [ ]:
dev = torch.device(DEVICE) if DEVICE else None

specs = [EvalRunSpec(**cfg) for cfg in EXPERIMENTS]
results: list[dict[str, Any]] = run_comparison_batch(
    specs,
    spair_root=SPAIR_ROOT,
    alphas=ALPHAS,
    device=dev,
)

for r in results:
    print("\n===", r["name"], "===")
    if "checkpoint_load" in r:
        print("checkpoint_load:", r["checkpoint_load"])
    for k, v in sorted(r["metrics"].items()):
        print(f"  {k}: {v:.4f}")

## Comparison table and bar chart

If **pandas** is installed (`pip install -e ".[notebook]"`), you get a sortable table. Otherwise a plain list of dicts is printed.

In [ ]:
rows = metrics_rows_for_table(results)

if pd is not None:
    df = pd.DataFrame(rows)
    display(df)
    metric_cols = [c for c in df.columns if c.startswith("pck@")]
    if metric_cols:
        df.plot.bar(x="name", y=metric_cols, figsize=(10, 4), rot=15)
        plt.ylabel("micro PCK")
        plt.tight_layout()
        plt.show()
else:
    print("Install pandas for DataFrame display: pip install pandas")
    for row in rows:
        print(row)

## Export results (JSON + optional CSV)

Files go to `runs/notebook_exports/` (gitignored). Reload later to compare without re-running evaluation.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_json = OUTPUT_DIR / "last_comparison.json"
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, default=str)
print("Wrote:", out_json)

if pd is not None and rows:
    out_csv = OUTPUT_DIR / "last_comparison.csv"
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    print("Wrote:", out_csv)

## Optional: load saved JSON and compare

Point `LOAD_JSON` at a file produced above to rebuild tables/plots without recomputing metrics.

In [ ]:
LOAD_JSON: Optional[str] = None  # e.g. str(OUTPUT_DIR / "last_comparison.json")

if LOAD_JSON and os.path.isfile(LOAD_JSON):
    with open(LOAD_JSON, encoding="utf-8") as f:
        loaded = json.load(f)
    rows_loaded = metrics_rows_for_table(loaded)
    if pd is not None:
        display(pd.DataFrame(rows_loaded))
    else:
        print(rows_loaded)
else:
    print("Set LOAD_JSON to a saved `last_comparison.json` path to use this cell.")